# OmniSQL-7B Baseline Evaluation — EHRSQL 2024 MIMIC-IV

Runs `seeklhy/OmniSQL-7B` with **K=10 hybrid retrieval** (BM25 + bge-large-en-v1.5) on the EHRSQL 2024 MIMIC-IV test set.

**Runtime:** A100 (40 GB or 80 GB). Model loads in bfloat16 (~14 GB VRAM), no quantization needed.

**Expected time:** ~25–35 min for 1,167 test questions.

### Files required in `/content/data/`
Upload or mount from Drive:
```
data/
  mimic_iv.sqlite
  train_combined_embeddings_bge_large.npy
  train/data.json  train/label.json
  train_aug/data.json  train_aug/label.json
  test/data.json   test/label.json
```

## 1 — Install dependencies

In [ ]:
!pip install -q rank_bm25 sentence-transformers transformers accelerate

## 2 — Upload files

**Option A — Upload from your computer** (run the cell below, then select the files from `colab/data/`):

**Option B — Mount Google Drive** if you've already stored the files there (uncomment the Drive cell instead).

In [ ]:
# Option A: upload files directly from your machine
# Run this cell, click 'Choose Files', and select everything inside colab/data/
# Then move them to the right subdirectories with the mv commands below.

import os
from google.colab import files

os.makedirs('/content/data/train', exist_ok=True)
os.makedirs('/content/data/train_aug', exist_ok=True)
os.makedirs('/content/data/test', exist_ok=True)

uploaded = files.upload()  # select all files at once

# Move each file to its correct location
import shutil
file_map = {
    'mimic_iv.sqlite':                         '/content/data/mimic_iv.sqlite',
    'train_combined_embeddings_bge_large.npy':  '/content/data/train_combined_embeddings_bge_large.npy',
    'train_data.json':   '/content/data/train/data.json',
    'train_label.json':  '/content/data/train/label.json',
    'train_aug_data.json':   '/content/data/train_aug/data.json',
    'train_aug_label.json':  '/content/data/train_aug/label.json',
    'test_data.json':    '/content/data/test/data.json',
    'test_label.json':   '/content/data/test/label.json',
}
for src, dst in file_map.items():
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f'  {src} → {dst}')

print('Upload complete.')

In [ ]:
# Option B: Mount Google Drive instead
# Uncomment and set DRIVE_DATA_PATH to wherever you stored the data/ folder on Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DATA_PATH = '/content/drive/MyDrive/ehrsql_data'   # <-- change this
# !cp -r {DRIVE_DATA_PATH}/. /content/data/
# print('Drive copy complete.')

## 3 — Verify files

In [ ]:
import os
required = [
    '/content/data/mimic_iv.sqlite',
    '/content/data/train_combined_embeddings_bge_large.npy',
    '/content/data/train/data.json',
    '/content/data/train/label.json',
    '/content/data/train_aug/data.json',
    '/content/data/train_aug/label.json',
    '/content/data/test/data.json',
    '/content/data/test/label.json',
]
all_ok = True
for path in required:
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else None
    status = f'{size_mb:.1f} MB' if size_mb else 'MISSING ❌'
    print(f'  {"✅" if size_mb else "❌"}  {path}  ({status})')
    if not size_mb:
        all_ok = False
print()
print('All files present ✅' if all_ok else 'Some files are missing — re-run upload cell ❌')

## 4 — Configuration

In [ ]:
from pathlib import Path

DATA_DIR      = Path('/content/data')
DB_PATH       = DATA_DIR / 'mimic_iv.sqlite'
TRAIN_DIR     = DATA_DIR / 'train'
TRAIN_AUG_DIR = DATA_DIR / 'train_aug'
TEST_DIR      = DATA_DIR / 'test'
EMBED_CACHE   = DATA_DIR / 'train_combined_embeddings_bge_large.npy'

MODEL_ID       = 'seeklhy/OmniSQL-7B'
EMBED_MODEL    = 'BAAI/bge-large-en-v1.5'
FEW_SHOT_K     = 10       # number of retrieved examples per question
MAX_NEW_TOKENS = 1024     # max SQL output length
ENABLE_REPAIR  = False    # set True to enable execution-guided repair loop
OUTPUT_FILE    = '/content/omnisql_7b_results.json'
PREDS_FILE     = '/content/omnisql_7b_preds.jsonl'

ABSTAIN = '[ABSTAIN]'
print('Config OK')

## 5 — Helpers (data loading, SQL post-processing, metrics)

In [ ]:
import json, re, sqlite3, time, math
import numpy as np
import torch

# ── Data loading ──────────────────────────────────────────────────────────────
def load_split(split_dir: Path) -> list[dict]:
    data   = json.load(open(split_dir / 'data.json'))['data']
    labels = json.load(open(split_dir / 'label.json'))
    out = []
    for ex in data:
        sql = labels.get(ex['id'], 'null')
        answerable = sql.strip().lower() not in ('null', 'none', 'n/a', '')
        out.append({'id': ex['id'], 'question': ex['question'],
                    'sql': sql if answerable else '', 'is_answerable': answerable})
    return out

# ── SQL post-processing (mirrors official EHRSQL 2024 scorer) ─────────────────
_CURRENT_TIME = '2100-12-31 23:59:00'
_CURRENT_DATE = '2100-12-31'
_CURRENT_TIME_ONLY = '23:59:00'
_TIME_PATTERN = re.compile(
    r'(DATE_SUB|DATE_ADD)\((\w+\(\)|\'[^\']+\')[, ]+INTERVAL (\d+) (MONTH|YEAR|DAY)\)',
    re.IGNORECASE)
_VITAL_RANGES = {
    'temperature': (35.5, 38.1), 'sao2': (95.0, 100.0), 'heart rate': (60.0, 100.0),
    'respiration': (12.0, 18.0), 'systolic bp': (90.0, 120.0),
    'diastolic bp': (60.0, 90.0), 'mean bp': (60.0, 110.0),
}
_VITAL_LOWER_RE = re.compile(r'[ \n]+([a-zA-Z0-9_]+_lower)')
_VITAL_UPPER_RE = re.compile(r'[ \n]+([a-zA-Z0-9_]+_upper)')

def _date_fn_to_sqlite(m):
    fn, date, n, unit = m.group(1).upper(), m.group(2), m.group(3), m.group(4).lower()
    unit = unit.rstrip('s') if n == '1' else (unit if unit.endswith('s') else unit + 's')
    sign = '-' if fn == 'DATE_SUB' else '+'
    return f"datetime({date}, '{sign}{n} {unit}')"

def post_process_sql(sql: str) -> str:
    sql = re.sub(r'[ ]+', ' ', sql.replace('\n', ' ')).strip()
    sql = sql.replace('> =', '>=').replace('< =', '<=').replace('! =', '!=')
    sql = _TIME_PATTERN.sub(_date_fn_to_sqlite, sql)
    for ph, val in [('current_time', f"'{_CURRENT_TIME}'"), ('current_date', f"'{_CURRENT_DATE}'"),
                    ("'now'", f"'{_CURRENT_TIME}'"), ('NOW()', f"'{_CURRENT_TIME}'"),
                    ('CURDATE()', f"'{_CURRENT_DATE}'"), ('CURTIME()', f"'{_CURRENT_TIME_ONLY}'")]:
        if ph in sql: sql = sql.replace(ph, val)
    lm, um = _VITAL_LOWER_RE.search(sql), _VITAL_UPPER_RE.search(sql)
    if lm and um:
        vnames = list(set(re.findall(r'([a-zA-Z0-9_]+)_lower', lm.group(1)) +
                          re.findall(r'([a-zA-Z0-9_]+)_upper', um.group(1))))
        if len(vnames) == 1:
            vk = vnames[0].replace('_', ' ')
            if vk in _VITAL_RANGES:
                lo, hi = _VITAL_RANGES[vk]
                sql = sql.replace(lm.group(1), str(lo)).replace(um.group(1), str(hi))
    return sql.replace('%y', '%Y').replace('%j', '%J')

def _norm_item(v) -> str:
    try: return str(round(float(v), 3))
    except: return str(v)

def _norm_result(rows) -> str:
    if not rows: return '[]'
    return str(sorted([[_norm_item(v) for v in r.values()] for r in rows])[:100])

def exec_sql(sql: str):
    try:
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        conn.row_factory = sqlite3.Row
        rows = [dict(r) for r in conn.execute(sql).fetchmany(100)]
        conn.close()
        return rows, None
    except Exception as e:
        return None, str(e)

print('Helpers loaded ✅')


## 6 — System prompt

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical analytics assistant. Convert the user's question into "
    "a valid SQLite SELECT query over the EHRSQL-MIMIC-IV database.\n\n"
    "Output exactly [ABSTAIN] — nothing else — when the question asks about "
    "information NOT derivable from the schema below. This includes: "
    "doctor or provider identities, family history, drug side effects or "
    "pharmacology, future or scheduled hospital visits, patient contact "
    "information, or any concept not represented by the tables and columns "
    "listed below.\n\n"
    "Otherwise output only the SQL query with no explanation.\n\n"
    "Database schema (EHRSQL-MIMIC-IV):\n"
    "  patients(row_id, subject_id, gender, dod)\n"
    "  admissions(row_id, subject_id, hadm_id, admittime, dischtime, "
    "admission_type, admission_location, discharge_location, insurance, "
    "language, marital_status, age)\n"
    "  d_icd_diagnoses(row_id, icd_code, long_title)\n"
    "  d_icd_procedures(row_id, icd_code, long_title)\n"
    "  d_labitems(row_id, itemid, label)\n"
    "  d_items(row_id, itemid, label, abbreviation, linksto)\n"
    "  diagnoses_icd(row_id, subject_id, hadm_id, icd_code, charttime)\n"
    "  procedures_icd(row_id, subject_id, hadm_id, icd_code, charttime)\n"
    "  labevents(row_id, subject_id, hadm_id, itemid, charttime, valuenum, valueuom)\n"
    "  chartevents(row_id, subject_id, hadm_id, stay_id, itemid, charttime, valuenum, valueuom)\n"
    "  prescriptions(row_id, subject_id, hadm_id, starttime, stoptime, drug, "
    "dose_val_rx, dose_unit_rx, route)\n"
    "  icustays(row_id, subject_id, hadm_id, stay_id, first_careunit, "
    "last_careunit, intime, outtime)\n"
    "  transfers(row_id, subject_id, hadm_id, transfer_id, eventtype, careunit, intime, outtime)\n"
    "  microbiologyevents(row_id, subject_id, hadm_id, charttime, "
    "spec_type_desc, test_name, org_name)\n"
    "  inputevents(row_id, subject_id, hadm_id, stay_id, starttime, itemid, totalamount, totalamountuom)\n"
    "  outputevents(row_id, subject_id, hadm_id, stay_id, charttime, itemid, value, valueuom)\n"
    "  cost(row_id, subject_id, hadm_id, event_type, event_id, chargetime, cost)\n"
    "Foreign keys: admissions.subject_id→patients.subject_id | "
    "icustays.hadm_id→admissions.hadm_id | icustays.subject_id→patients.subject_id | "
    "diagnoses_icd.hadm_id→admissions.hadm_id | diagnoses_icd.icd_code=d_icd_diagnoses.icd_code | "
    "procedures_icd.hadm_id→admissions.hadm_id | procedures_icd.icd_code=d_icd_procedures.icd_code | "
    "labevents.hadm_id→admissions.hadm_id | labevents.itemid→d_labitems.itemid | "
    "prescriptions.hadm_id→admissions.hadm_id | chartevents.stay_id→icustays.stay_id | "
    "chartevents.itemid→d_items.itemid | microbiologyevents.hadm_id→admissions.hadm_id | "
    "inputevents.stay_id→icustays.stay_id | inputevents.itemid→d_items.itemid | "
    "outputevents.stay_id→icustays.stay_id | outputevents.itemid→d_items.itemid | "
    "cost.hadm_id→admissions.hadm_id | transfers.hadm_id→admissions.hadm_id"
)
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

## 7 — Build hybrid retriever (K=10)

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

print('Loading train + train_aug corpus ...')
train_all = [e for e in load_split(TRAIN_DIR)     if e['is_answerable']]
aug_all   = [e for e in load_split(TRAIN_AUG_DIR) if e['is_answerable']]
corpus    = train_all + aug_all
print(f'  Corpus: {len(corpus):,} examples')

def _sql_skeleton(sql: str) -> str:
    s = re.sub(r"'[^']*'", "'X'", sql.lower().strip())
    s = re.sub(r'\b\d+(?:\.\d+)?\b', 'N', s)
    return re.sub(r'\s+', ' ', s).strip()

print('Building BM25 index ...')
bm25 = BM25Okapi([ex['question'].lower().split() for ex in corpus])
n = len(corpus)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading embedding model on {device} ...')
embed_model = SentenceTransformer(EMBED_MODEL, device=device)

if EMBED_CACHE.exists():
    train_embeds = np.load(str(EMBED_CACHE)).astype(np.float32)
    if train_embeds.shape[0] == n:
        print(f'  Loaded cached embeddings {train_embeds.shape}')
    else:
        print(f'  Cache shape mismatch ({train_embeds.shape[0]} vs {n}) — rebuilding ...')
        train_embeds = None
else:
    print('  No cache found — computing embeddings (~10 min) ...')
    train_embeds = None

if train_embeds is None:
    index_texts  = [ex['question'] + ' ' + _sql_skeleton(ex['sql']) for ex in corpus]
    train_embeds = embed_model.encode(index_texts, show_progress_bar=True,
                                      batch_size=64, normalize_embeddings=True).astype(np.float32)
    np.save(str(EMBED_CACHE), train_embeds)
    print(f'  Saved to {EMBED_CACHE}')

train_embeds_t = torch.from_numpy(train_embeds).to(device)

def retrieve(question: str, k: int = FEW_SHOT_K) -> str:
    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_order  = (-bm25_scores).argsort()
    bm25_ranks  = np.empty(n, dtype=np.float32)
    bm25_ranks[bm25_order] = np.arange(1, n + 1, dtype=np.float32)

    q_vec = embed_model.encode([question], normalize_embeddings=True)
    q_t   = torch.from_numpy(q_vec.astype(np.float32)).to(device)
    cos   = (train_embeds_t @ q_t.T).squeeze().cpu().numpy()
    embed_order = (-cos).argsort()
    embed_ranks = np.empty(n, dtype=np.float32)
    embed_ranks[embed_order] = np.arange(1, n + 1, dtype=np.float32)

    rrf = 1.0 / (60 + bm25_ranks) + 1.0 / (60 + embed_ranks)
    top = list(map(int, (-rrf).argsort()[:k]))

    lines = ['Similar examples:']
    for i in top:
        lines.append(f"Q: {corpus[i]['question']}")
        lines.append(f"SQL: {corpus[i]['sql']}")
    return '\n'.join(lines)

print('Retriever ready ✅')

## 8 — Load OmniSQL-7B (bfloat16)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading {MODEL_ID} in bfloat16 ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'   # required for batched generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded ✅  VRAM used: {mem_gb:.1f} GB')

BATCH_SIZE = 4   # increase to 8 if VRAM allows

def _strip_fences(text: str) -> str:
    m = re.search(r'```(?:sql)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

def generate_batch(messages_list: list) -> list:
    """Generate SQL for a batch of message lists in one forward pass."""
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in messages_list
    ]
    inputs = tokenizer(
        prompts, return_tensors='pt', padding=True, truncation=False
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    results = []
    for out in outputs:
        text = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        results.append(_strip_fences(text))
    return results


## 9 — Run evaluation

In [ ]:
test_all = load_split(TEST_DIR)
print(f'Test set: {len(test_all)} questions '
      f'({sum(e["is_answerable"] for e in test_all)} answerable, '
      f'{sum(not e["is_answerable"] for e in test_all)} unanswerable)')

# Precompute all test question embeddings at once — eliminates 1167 per-question GPU calls
print('Precomputing test question embeddings ...')
test_questions = [ex['question'] for ex in test_all]
test_q_embeds  = embed_model.encode(
    test_questions, show_progress_bar=True, batch_size=128, normalize_embeddings=True
)
test_q_embeds_t = torch.from_numpy(test_q_embeds.astype(np.float32)).to(device)
print(f'  Done: {test_q_embeds_t.shape}')

def retrieve_precomputed(q_embed_t, k: int = FEW_SHOT_K) -> str:
    """Retrieve using a precomputed query embedding (no model call needed)."""
    bm25_scores = bm25.get_scores(test_questions[0].lower().split())  # placeholder for BM25 (overridden below)
    cos = (train_embeds_t @ q_embed_t.unsqueeze(1)).squeeze().cpu().numpy()
    embed_order = (-cos).argsort()
    embed_ranks = np.empty(n, dtype=np.float32)
    embed_ranks[embed_order] = np.arange(1, n + 1, dtype=np.float32)
    return embed_ranks

def retrieve_hybrid_precomputed(question: str, q_embed_t, k: int = FEW_SHOT_K) -> str:
    bm25_scores = bm25.get_scores(question.lower().split())
    bm25_order  = (-bm25_scores).argsort()
    bm25_ranks  = np.empty(n, dtype=np.float32)
    bm25_ranks[bm25_order] = np.arange(1, n + 1, dtype=np.float32)

    cos = (train_embeds_t @ q_embed_t.unsqueeze(1)).squeeze().cpu().numpy()
    embed_order = (-cos).argsort()
    embed_ranks = np.empty(n, dtype=np.float32)
    embed_ranks[embed_order] = np.arange(1, n + 1, dtype=np.float32)

    rrf = 1.0 / (60 + bm25_ranks) + 1.0 / (60 + embed_ranks)
    top = list(map(int, (-rrf).argsort()[:k]))

    lines = ['Similar examples:']
    for i in top:
        lines.append(f"Q: {corpus[i]['question']}")
        lines.append(f"SQL: {corpus[i]['sql']}")
    return '\n'.join(lines)

# ── Evaluation loop (batched) ──────────────────────────────────────────────────
correct_answers     = 0
wrong_abstentions   = 0
wrong_sql_ans       = 0
wrong_on_unans      = 0
correct_abstentions = 0
gold_exec_valid     = 0
gold_exec_empty     = 0
gold_exec_error     = 0
latencies           = []
preds_log           = []

preds_fh = open(PREDS_FILE, 'w')
t_loop_start = time.time()

for batch_start in range(0, len(test_all), BATCH_SIZE):
    batch     = test_all[batch_start : batch_start + BATCH_SIZE]
    batch_idx = list(range(batch_start, batch_start + len(batch)))
    t0 = time.time()

    # Build messages for each item in batch
    msgs_list = []
    for i, ex in enumerate(batch):
        q        = ex['question']
        q_embed  = test_q_embeds_t[batch_start + i]
        examples = retrieve_hybrid_precomputed(q, q_embed)
        user_content = f"{examples}\n\nQuestion: {q}"
        msgs_list.append([
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_content},
        ])

    predictions = generate_batch(msgs_list)
    elapsed_ms  = (time.time() - t0) * 1000
    per_q_ms    = elapsed_ms / len(batch)

    for i, (ex, predicted) in enumerate(zip(batch, predictions)):
        latencies.append(per_q_ms)
        abstained = (predicted == ABSTAIN or not predicted)

        # Optional repair (single question at a time)
        if ENABLE_REPAIR and not abstained:
            _, err = exec_sql(post_process_sql(predicted))
            if err:
                repair_msgs = msgs_list[i] + [
                    {'role': 'assistant', 'content': predicted},
                    {'role': 'user', 'content': (
                        f'SQLite error: {err}\n'
                        'Fix the SQL using only the schema above. Output only the corrected SQL.'
                    )},
                ]
                repaired_list = generate_batch([repair_msgs])
                repaired = repaired_list[0]
                _, new_err = exec_sql(post_process_sql(repaired))
                if new_err is None:
                    predicted = repaired

        outcome = 'unknown'
        if ex['is_answerable']:
            gold_rows, gold_err = exec_sql(post_process_sql(ex['sql']))
            if gold_err:        gold_exec_error += 1
            elif not gold_rows: gold_exec_empty += 1
            else:               gold_exec_valid += 1

            if abstained:
                wrong_abstentions += 1; outcome = 'wrong_abstention'
            else:
                pred_rows, pred_err = exec_sql(post_process_sql(predicted))
                if pred_err is None and _norm_result(pred_rows) == _norm_result(gold_rows):
                    correct_answers += 1; outcome = 'correct'
                else:
                    wrong_sql_ans += 1; outcome = 'wrong_sql'
        else:
            if abstained:
                correct_abstentions += 1; outcome = 'correct_abstention'
            else:
                wrong_on_unans += 1; outcome = 'hallucination'

        row = {'id': ex['id'], 'question': ex['question'], 'gold_sql': ex['sql'],
               'predicted_sql': predicted, 'outcome': outcome, 'latency_ms': round(per_q_ms, 1)}
        preds_log.append(row)
        preds_fh.write(json.dumps(row) + '\n')

    preds_fh.flush()
    done    = batch_start + len(batch)
    rs10    = (correct_answers + correct_abstentions - 10 * wrong_on_unans) / done
    elapsed_total = time.time() - t_loop_start
    eta_min = (elapsed_total / done) * (len(test_all) - done) / 60
    print(f'[{done:4d}/{len(test_all)}]  '
          f'correct={correct_answers}  hall={wrong_on_unans}  '
          f'RS(10)={rs10:.3f}  {per_q_ms:.0f}ms/q  ETA≈{eta_min:.1f}min')

preds_fh.close()
print('\nEvaluation complete ✅')


## 10 — Results

In [ ]:
total = len(test_all)
ans_total = sum(e['is_answerable'] for e in test_all)

lat_sorted = sorted(latencies)
results = {
    'model': MODEL_ID, 'few_shot_k': FEW_SHOT_K, 'repair': ENABLE_REPAIR,
    'total': total, 'answerable': ans_total, 'unanswerable': total - ans_total,
    'EX':     round(correct_answers / max(1, ans_total), 4),
    'RS(0)':  round((correct_answers + correct_abstentions) / total, 4),
    'RS(5)':  round((correct_answers + correct_abstentions - 5  * wrong_on_unans) / total, 4),
    'RS(10)': round((correct_answers + correct_abstentions - 10 * wrong_on_unans) / total, 4),
    'correct_answers':               correct_answers,
    'wrong_abstentions':             wrong_abstentions,
    'wrong_sql_answerable':          wrong_sql_ans,
    'wrong_answers_on_unanswerable': wrong_on_unans,
    'correct_abstentions':           correct_abstentions,
    'gold_exec_valid': gold_exec_valid,
    'gold_exec_empty': gold_exec_empty,
    'gold_exec_error': gold_exec_error,
    'p50_latency_ms': round(lat_sorted[len(lat_sorted) // 2], 1),
    'p95_latency_ms': round(lat_sorted[int(len(lat_sorted) * 0.95)], 1),
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print('=' * 50)
print('  OmniSQL-7B — EHRSQL 2024 MIMIC-IV')
print('=' * 50)
for k in ['EX', 'RS(0)', 'RS(5)', 'RS(10)']:
    print(f'  {k:8s}: {results[k]:.4f}')
print()
for k in ['correct_answers', 'wrong_abstentions', 'wrong_sql_answerable',
          'wrong_answers_on_unanswerable', 'correct_abstentions']:
    print(f'  {k}: {results[k]}')
print(f'\n  Latency p50={results["p50_latency_ms"]}ms  p95={results["p95_latency_ms"]}ms')
print(f'\n  Results → {OUTPUT_FILE}')
print(f'  Predictions → {PREDS_FILE}')

## 11 — Download results

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)
files.download(PREDS_FILE)